In [6]:
import pandas as pd
import numpy as np

def fill_solar_site_metadata(df):
    """
    Xử lý metadata solar site theo chiến lược cụ thể:
    - capacity_kw, number_of_panels: Campus Median + Flag.
    - panel, inverter, location_name: 'Unknown'.
    - optimizers: 'None'.
    - site_metric: 'kWh'.
    """
    # Xử lý campus_name trước để làm key tính median
    df['campus_name'] = df.groupby('site_id')['campus_name'].transform(lambda x: x.ffill().bfill())
    df['campus_name'] = df['campus_name'].fillna('Unknown')

    # capacity_kw: Campus Median + Flag
    if 'capacity_kw' in df.columns:
        mask = df['capacity_kw'].isnull()
        df['capacity_kw_is_imputed'] = mask.astype(int)
        df['capacity_kw'] = df['capacity_kw'].fillna(df.groupby('campus_name')['capacity_kw'].transform('median'))
        df['capacity_kw'] = df['capacity_kw'].fillna(df['capacity_kw'].median())

    # number_of_panels: Campus Median + Flag
    if 'number_of_panels' in df.columns:
        mask = df['number_of_panels'].isnull()
        df['number_of_panels_is_imputed'] = mask.astype(int)
        df['number_of_panels'] = df['number_of_panels'].fillna(df.groupby('campus_name')['number_of_panels'].transform('median'))
        df['number_of_panels'] = df['number_of_panels'].fillna(df['number_of_panels'].median())

    # panel, inverter, location_name -> Unknown
    for col in ['panel', 'inverter', 'location_name']:
        if col in df.columns:
            df[col] = df.groupby('site_id')[col].transform(lambda x: x.ffill().bfill())
            df[col] = df[col].fillna('Unknown')

    # optimizers -> None
    if 'optimizers' in df.columns:
        df['optimizers'] = df['optimizers'].fillna('None')

    # site_metric -> kWh
    if 'site_metric' in df.columns:
        df['site_metric'] = df['site_metric'].fillna('kWh')
        
    return df

def fill_geo_coordinates(df, geo_cols):
    for col in geo_cols:
        if col in df.columns:
            df[col] = df.groupby('site_id')[col].transform(lambda x: x.ffill().bfill())
    return df

def fill_weather_metadata(df, cols):
    if 'weather_timestamp' in df.columns:
        df['weather_timestamp'] = pd.to_datetime(df['weather_timestamp'], errors='coerce')
        df.loc[df['weather_timestamp'].dt.year < 2019, 'weather_timestamp'] = pd.NaT

    ID_COLS = ['weather_id']
    for col in cols:
        if col not in df.columns: continue
        if col == 'weather_id': continue
        
        if col == 'weather_type_id':
            df[col] = df.groupby('site_id')[col].transform(lambda x: x.ffill().bfill())
            df[col] = df[col].fillna(df.groupby(['month_tmp', 'hour_tmp'])[col].transform(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan))
            df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else np.nan)
        elif col in ['weather_condition', 'weather_description']:
            df[col] = df.groupby('site_id')[col].transform(lambda x: x.ffill())
            df[col] = df[col].fillna('Unknown')
        elif col == 'weather_code':
            df[col] = df.groupby('site_id')[col].transform(lambda x: x.ffill().bfill())
    return df

def fill_weather_is_day(df):
    if 'weather_is_day' not in df.columns: return df
    is_day_time = ((df['hour_tmp'] >= 6) & (df['hour_tmp'] <= 18)).astype(int)
    df['weather_is_day'] = df['weather_is_day'].fillna(is_day_time)
    
    rad_cols = ['shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation', 'sunshine_duration']
    for col in rad_cols:
        if col in df.columns:
            mask = df['weather_is_day'].isnull()
            df.loc[mask, 'weather_is_day'] = (df.loc[mask, col] > 0).astype(int)
    df['weather_is_day'] = df['weather_is_day'].fillna(0)
    return df

def fill_weather_metrics_advanced(df):
    df = df.sort_values(by=['site_id', 'timestamp']).set_index('timestamp')
    night_mask = (df['hour_tmp'] < 5.5) | (df['hour_tmp'] >= 18.5)
    
    # Zero-fill night
    for col in ['shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation', 'sunshine_duration',
                'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high']:
        if col in df.columns:
            df.loc[df[col].isnull() & night_mask, col] = 0.0

    # Interpolate
    if 'temperature_c' in df.columns:
        mask = df['temperature_c'].isnull()
        df['temperature_c_is_imputed'] = mask.astype(int)
        df['temperature_c'] = df.groupby('site_id')['temperature_c'].transform(lambda x: x.interpolate(method='time', limit=12))
        df['temperature_c'] = df['temperature_c'].fillna(df.groupby(['site_id', 'hour_tmp'])['temperature_c'].transform('median'))
        df['temperature_c'] = df['temperature_c'].fillna(df['temperature_c'].median())

    if 'wind_speed' in df.columns:
        mask = df['wind_speed'].isnull()
        df['wind_speed_is_imputed'] = mask.astype(int)
        df['wind_speed'] = df.groupby('site_id')['wind_speed'].transform(lambda x: x.interpolate(method='time', limit=4))
        df['wind_speed'] = df['wind_speed'].fillna(df.groupby(['site_id', 'hour_tmp'])['wind_speed'].transform('median'))
        df['wind_speed'] = df['wind_speed'].fillna(df['wind_speed'].median())

    # Cloud fill
    for col in ['cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high']:
        if col in df.columns:
            mask = df[col].isnull()
            df[f'{col}_is_imputed'] = mask.astype(int)
            df[col] = df.groupby('site_id')[col].transform(lambda x: x.ffill(limit=8))
            df[col] = df[col].fillna(df.groupby(['site_id', 'hour_tmp'])[col].transform('median'))
            df[col] = df[col].fillna(df[col].median())

    if 'precipitation_mm' in df.columns:
        mask = df['precipitation_mm'].isnull()
        df['precipitation_mm_is_imputed'] = mask.astype(int)
        df['precipitation_mm'] = df['precipitation_mm'].fillna(0.0)

    return df.reset_index()

def run_pipeline(input_path, output_path):
    df_raw = pd.read_parquet(input_path)
    df_working = df_raw.copy()
    
    df_working['timestamp'] = pd.to_datetime(df_working['timestamp'])
    df_working['month_tmp'] = df_working['timestamp'].dt.month
    df_working['hour_tmp'] = df_working['timestamp'].dt.hour
    
    solar_meta = ['campus_name', 'capacity_kw', 'number_of_panels', 'panel', 'inverter', 'optimizers', 'site_metric', 'location_name']
    geo_cols = ['latitude', 'longitude']
    weather_meta = ['weather_id', 'weather_type_id', 'weather_timestamp', 'weather_is_day', 'weather_code', 'weather_condition', 'weather_description']
    
    df_working = fill_geo_coordinates(df_working, geo_cols)
    df_working = fill_solar_site_metadata(df_working)
    df_working = fill_weather_metadata(df_working, weather_meta)
    df_working = fill_weather_is_day(df_working)
    df_working = fill_weather_metrics_advanced(df_working)
    
    df_working = df_working.drop(columns=['month_tmp', 'hour_tmp'])
    df_working.to_parquet(output_path, index=False)




In [9]:
run_pipeline("../../data/mlmart_base/v3_preprocessing.parquet", "../../data/mlmart_base/v3_final_cleaned.parquet")